<a href="https://colab.research.google.com/github/IBM/vLLM-Hook/blob/main/notebooks/demo_corer_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Core Reranker
vLLM-Hook is an extensible framework that aims to allow selective access to model internals during the inference. 
As a demonstration of that, in this notebook, we show how vLLM-Hook enables *Core Reranker* for document relevance scoring. 

**Paper**: [Contrastive Retrieval Heads Improve Attention-Based Re-Ranking](https://arxiv.org/abs/2510.02219).<br />
**Authors**: Linh Tran, Yulong Li, Radu Florian, Wei Sun <br />
**"TL;DR"**: Core reranker is an attention-based reranker that leverage attention weights from selected transformer heads to produce document relevance scores.


### Installation
If running this from a new environment, please use the cell below to install `vllm_hook_plugins`. Update the path/command to match your environment.<br />
The following block is not necessary if running this notebook from an environment where the package has already been installed.

In [1]:
from pathlib import Path
import importlib
import importlib.util
import os
import re
import shutil
import site
import subprocess
import sys
import textwrap # Import textwrap to dedent the string

REPO_URL = "https://github.com/IBM/vLLM-Hook.git"
REPO_BRANCH = "main"
REPO_NAME = "vLLM-Hook"
COLAB_INSTALL_VLLM = os.environ.get("COLAB_INSTALL_VLLM", "")
VLLM_SPEC = os.environ.get("VLLM_SPEC", "vllm>=0.11,<0.19") # Reverted to original flexible range
VLLM_TORCH_BACKEND = os.environ.get("VLLM_TORCH_BACKEND", "cu128") # Corrected to cu128

IN_COLAB = "google.colab" in sys.modules
NOTEBOOK_DIR = Path.cwd()


def run(cmd, cwd=None, env=None):
    cmd = [str(part) for part in cmd]
    print("Running:", " ".join(cmd), flush=True)
    process = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        tail = tail[-120:]
    returncode = process.wait()
    if returncode:
        tail_text = "\n".join(tail)
        raise RuntimeError(
            f"Command failed with exit code {returncode}: {' '.join(cmd)}\n\n"
            f"Last output lines:\n{tail_text}"
        )


def run_capture(cmd):
    return subprocess.run([str(part) for part in cmd], text=True, capture_output=True, check=False)


def norm(name):
    return name.lower().replace("_", "-")


def package_from_req_line(line: str) -> str:
    stripped = line.strip()
    package = re.split(r"==|>=|<=|~=|!=|<|>|\[", stripped, maxsplit=1)[0]
    return norm(package.strip())


def _repo_remote_matches(repo_root: Path, expected_remote: str) -> bool:
    try:
        origin_url = subprocess.run(
            ["git", "-C", str(repo_root), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip().removesuffix(".git")
    except Exception:
        return False
    return origin_url == expected_remote


def _find_existing_repo_root(start_dir: Path, expected_remote: str):
    for candidate in [start_dir, *start_dir.parents]:
        if (candidate / ".git").exists() and _repo_remote_matches(candidate, expected_remote):
            return candidate
    return None


def assert_cuda_runtime():
    try:
        import torch
    except Exception:
        torch = None
    has_cuda = bool(torch is not None and torch.cuda.is_available())
    has_cudart = importlib.util.find_spec("nvidia.cuda_runtime") is not None
    if not has_cuda and not has_cudart:
        raise RuntimeError(
            "This notebook requires a Colab GPU runtime with CUDA available. "
            "In Colab, use Runtime > Change runtime type > T4 GPU (or another GPU), then rerun from a fresh runtime."
        )


def subprocess_smoke_check():
    current_plugin_src_dir = str(PKG_DIR.resolve()) # Get the plugin_src_dir

    code = textwrap.dedent(f'''
    import sys
    sys.path.insert(0, "{current_plugin_src_dir}") # Add plugin path to subprocess sys.path

    import importlib.metadata as md
    import inspect

    print("torch", md.version("torch"))
    print("vllm", md.version("vllm"))
    print("vllm-hook-plugins", md.version("vllm-hook-plugins"))

    import torch
    import vllm
    from vllm.engine.arg_utils import EngineArgs
    import vllm_hook_plugins # This import should now work

    fields = getattr(EngineArgs, "__dataclass_fields__", {{}})
    params = inspect.signature(EngineArgs).parameters
    if "worker_extension_cls" not in fields and "worker_extension_cls" not in params:
        raise RuntimeError("Installed vLLM does not support worker_extension_cls")

    print("smoke check OK")
    ''')
    run([sys.executable, "-c", code])


if IN_COLAB:
    expected_remote = REPO_URL.removesuffix(".git")
    existing_repo_root = _find_existing_repo_root(NOTEBOOK_DIR, expected_remote)
    if existing_repo_root is not None:
        REPO_ROOT = existing_repo_root
        print(f"Colab detected. Reusing existing repo at {REPO_ROOT}")
    else:
        REPO_ROOT = Path("/content") / REPO_NAME
        if not REPO_ROOT.exists():
            print(f"Colab detected. Cloning {REPO_URL} ({REPO_BRANCH}) into {REPO_ROOT} ...")
            run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
        elif not _repo_remote_matches(REPO_ROOT, expected_remote):
            print(f"Remote mismatch under {REPO_ROOT}; replacing clone with {expected_remote}")
            shutil.rmtree(REPO_ROOT)
            run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
        else:
            print(f"Colab detected. Reusing existing clone at {REPO_ROOT}")
            print("Refreshing existing clone ...")
            run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH])
            run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH])
            run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH])
    NOTEBOOK_DIR = REPO_ROOT / "notebooks"
    os.chdir(NOTEBOOK_DIR)
    print(f"Changed working directory to {NOTEBOOK_DIR}")
else:
    REPO_ROOT = NOTEBOOK_DIR.parent

PKG_DIR = REPO_ROOT / "vllm_hook_plugins"
REQ_FILE = REPO_ROOT / "requirement.txt"
FILTERED_REQ_FILE = Path("/tmp/vllm_hook_colab_requirements.txt")
COLAB_RESTART_MARKER = Path("/tmp/vllm_hook_colab_binary_deps_restarted")

print("Colab      :", IN_COLAB)
print("Notebook dir:", NOTEBOOK_DIR)
print("Repo root   :", REPO_ROOT)
print("Repo branch :", REPO_BRANCH)
print("Package dir :", PKG_DIR)
print("Req file    :", REQ_FILE)

if IN_COLAB:
    assert_cuda_runtime()

if not PKG_DIR.exists():
    raise FileNotFoundError(f"Plugin directory not found: {PKG_DIR}")

if shutil.which("git") is None and IN_COLAB:
    raise RuntimeError("Colab was detected but git is unavailable in the runtime.")

if REQ_FILE.exists():
    keep = []
    blocked = {"vllm", "torch", "torchvision", "torchaudio", "numpy", "scipy", "protobuf"}
    for line in REQ_FILE.read_text().splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            keep.append(line)
            continue
        package = package_from_req_line(stripped)
        if package in blocked:
            print("Skipping requirement managed by Colab install cell:", line)
            continue
        keep.append(line)
    FILTERED_REQ_FILE.write_text("\n".join(keep) + "\n")
    run([sys.executable, "-m", "pip", "install", "-r", str(FILTERED_REQ_FILE)])
else:
    print("Warning: requirement.txt not found; skipping dependency install.")

run([sys.executable, "-m", "pip", "install", "--force-reinstall", "protobuf>=5.29.6,<6.30"])

if COLAB_INSTALL_VLLM:
    run([sys.executable, "-m", "pip", "install", COLAB_INSTALL_VLLM])
else:
    # Colab currently does not support CUDA 13 on all GPU runtimes, so use CUDA 12.x wheels.
    run([sys.executable, "-m", "pip", "uninstall", "-y", "vllm", "torch", "torchvision", "torchaudio"])
    for site_dir in site.getsitepackages():
        site_path = Path(site_dir)
        for leftover in [
            site_path / "vllm",
            *site_path.glob("vllm-*.dist-info"),
            site_path / "torch",
            *site_path.glob("torch-*.dist-info"),
            site_path / "torchvision",
            *site_path.glob("torchvision-*.dist-info"),
            site_path / "torchaudio",
            *site_path.glob("torchaudio-*.dist-info"),
        ]:
            if leftover.exists():
                print("Removing leftover:", leftover)
                shutil.rmtree(leftover) if leftover.is_dir() else leftover.unlink()


    # Force clear GPU memory and reload torch
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
    except:
        pass

    run([sys.executable, "-m", "pip", "install", "-U", "uv"])
    run([
        "uv", "pip", "install",
        "--system",
        VLLM_SPEC,
        "torch", "torchvision", "torchaudio",
        f"--torch-backend={VLLM_TORCH_BACKEND}",
    ])

scipy_check = run_capture([
    sys.executable,
    "-c",
    "import numpy, scipy; print('numpy', numpy.__version__); print('scipy', scipy.__version__)",
])
if scipy_check.returncode:
    print(scipy_check.stdout)
    print(scipy_check.stderr)
    run([sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall", "numpy", "scipy"])

run([
    sys.executable,
    "-c",
    "import torch, vllm; print('torch', torch.__version__); print('vllm', getattr(vllm, '__version__', 'unknown'))",
])

# FIXED (Skip pip install, just add path)
# Only run pip install if setup.py is necessary for dependencies
print("Skipping pip install -e to avoid compilation/OOM risks. Adding to sys.path directly...")
plugin_src_dir = str(PKG_DIR.resolve())
if plugin_src_dir not in sys.path:
    sys.path.insert(0, plugin_src_dir)
importlib.invalidate_caches()

#Verify plugin loads correctly without pip install -e
try:
    # Only attempt to import if the plugin doesn't require compilation
    import importlib.util
    spec = importlib.util.spec_from_file_location("vllm_hook_plugins", PKG_DIR / "__init__.py")
    if spec:
        importlib.util.module_from_spec(spec)
        print("Plugin module loaded successfully from sys.path.")
# Removed the `except` block because we *need* pip install -e to ensure metadata is present for subprocesses.
finally:
    # Unconditionally install in editable mode to ensure metadata is present for importlib.metadata
    run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(PKG_DIR)])

print("Plugin source:", plugin_src_dir)
print("Python exec  :", sys.executable)

if IN_COLAB and not COLAB_RESTART_MARKER.exists():
    COLAB_RESTART_MARKER.write_text("1\n")
    print("Restarting Colab runtime once so Python reloads the replaced vLLM/Torch binary packages.")
    # Add a small sleep to allow output buffer to flush before killing
    import time
    time.sleep(1)
    # os.kill(os.getpid(), 9) # Commented out as requested

subprocess_smoke_check()


### Importing the Hook-Enabled LLM
The plugin provides its own LLM wrapper that behaves like vllm.LLM (`from vllm import LLM`) but adds support for hooks and instrumentation.
We import it here:

In [2]:
from vllm import SamplingParams
from vllm_hook_plugins import HookLLM

### Environment & multiprocessing setup

In [3]:
import io
import os
import multiprocessing as mp
import sys
import torch
from typing import List

IN_COLAB = "google.colab" in sys.modules
os.environ["VLLM_USE_V1"] = "1"

if IN_COLAB:
    mp.set_start_method("fork", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "fork"
    os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
    os.environ.setdefault("HF_HOME", "/content/.cache/huggingface")
    os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/content/.cache/huggingface/hub")
    os.makedirs(os.environ["HUGGINGFACE_HUB_CACHE"], exist_ok=True)
    os.makedirs("/content/.cache/vllm-hook", exist_ok=True)

    def _patch_fileno(stream, fallback_stream, fallback_fd):
        try:
            stream.fileno()
        except io.UnsupportedOperation:
            def _fileno():
                try:
                    return fallback_stream.fileno()
                except Exception:
                    return fallback_fd
            stream.fileno = _fileno

    _patch_fileno(sys.stdout, sys.__stdout__, 1)
    _patch_fileno(sys.stderr, sys.__stderr__, 2)
    print("Colab detected: using fork start method and disabled V1 multiprocessing")
else:
    mp.set_start_method("spawn", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"


### Helper functions that give the instruction range
As Core Reranker needs to locate the candidate passages and the user query in the prompt, below is a helper function that gives the data range with texts.<br />
Check [Core Reranker](https://arxiv.org/pdf/2510.02219) for more details.

In [4]:
def apply_chat_template_and_get_ranges(tokenizer, model_name: str, query: str, documents: List[str]):
    # setup prompts
    off_set = 0
    if 'granite' in model_name.lower():
        prompt_prefix = '<|start_of_role|>user<|end_of_role|>'
        prompt_suffix = '<|end_of_text|><|start_of_role|>assistant<|end_of_role|>'
    elif 'llama' in model_name.lower():
        prompt_prefix = '<|start_header_id|>user<|end_header_id|>'
        prompt_suffix = '<|eot_id|><|start_header_id|>assistant<|end_header_id|>'
    elif 'mistral' in model_name.lower():
        prompt_prefix = '[INST]'
        prompt_suffix = '[/INST]'
        off_set = 1
    elif 'phi' in model_name.lower():
        prompt_prefix = '<|im_start|>user<|im_sep|>'
        prompt_suffix = '<|im_end|><|im_start|>assistant<|im_sep|>'
    retrieval_instruction = ' Here are some paragraphs:\n\n'
    retrieval_instruction_late = 'Please find information that are relevant to the following query in the paragraphs above.\n\nQuery: '
    
    doc_span = []
    query_start_idx = None
    query_end_idx = None

    llm_prompt = prompt_prefix + retrieval_instruction

    for i, doc in enumerate(documents):

        llm_prompt += f'[document {i+1}]'
        start_len = len(tokenizer(llm_prompt).input_ids)

        llm_prompt += ' ' + " ".join(doc)
        end_len = len(tokenizer(llm_prompt).input_ids) - off_set

        doc_span.append((start_len, end_len))
        llm_prompt += '\n\n'

    start_len = len(tokenizer(llm_prompt).input_ids)

    llm_prompt += retrieval_instruction_late
    after_retrieval_instruction_late = len(tokenizer(llm_prompt).input_ids) - off_set

    llm_prompt += f'{query.strip()}'
    end_len = len(tokenizer(llm_prompt).input_ids) - off_set
    llm_prompt += prompt_suffix

    query_start_idx = start_len
    query_end_idx = end_len

    return llm_prompt, (doc_span, query_start_idx, after_retrieval_instruction_late, query_end_idx)

### Initialize `HookLLM`
Before we create the LLM instance, we need to specify the model and data type:

In [5]:
cache_dir = '/content/.cache/vllm-hook' if IN_COLAB else os.path.expanduser('~/.cache/vllm-hook')
model = 'RedHatAI/granite-3.1-2b-instruct-quantized.w4a16'
    
dtype_map = {
    'RedHatAI/granite-3.1-2b-instruct-quantized.w4a16': torch.float16,
    'mistralai/Mistral-7B-Instruct-v0.3': torch.float16,
}


We also need to provide a config file that specifies the important heads we want to track. <br />
For Core Reranker, this config file can be obtained from [head_detection.py](https://github.com/linhhtran/CoRe-Reranking/blob/main/experiments/head_detection.py). 

In [6]:
import json
from pathlib import Path

json_path = Path("../model_configs/core_reranker/granite-3.1-2b-instruct-quantized.w4a16.json")

with open(json_path, "r") as f:
    config = json.load(f)

# print(config)


Inside `probe_hook_qk` and `core_reranker` we defined the desired behavior during model inference and after the model inference: 
- `workers/probe_hookqk_worker.py` defines that we need `q` (query) and `k` (key) to be saved during forward passes
- `analyzers/core_reranker_analyzer.py` calculates the passage relevance score and the final ranking of passages

Now, we initialize the llm:

Before rebuilding `HookLLM`, clear any stale CUDA allocations left by earlier notebook runs in the same Colab session.

In [ ]:
# Clear any prior HookLLM instance before rebuilding in the same Colab session.
import gc

try:
    del llm
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [7]:
llm = HookLLM(
    model=model,
    worker_name="probe_hook_qk",
    analyzer_name="core_reranker",
    config_file=json_path,
    download_dir=cache_dir,
    trust_remote_code=True,
    dtype=dtype_map[model],
    enable_hook=True,

    ### Memory Conservation Args (Important for Colab T4 instance)

    # Raise GPU reservation enough to leave room for model weights and a small KV cache.
    gpu_memory_utilization=0.3,

    # Cap context length to fit KV cache on the free public Colab T4 instance.
    max_model_len=1024,

    # Limit batch concurrency because a single free public Colab T4 has tight memory headroom.
    max_num_seqs=1,

    # Keep eager execution explicit so CUDA graph capture does not add memory overhead.
    enforce_eager=True,

    # Keep compilation settings conservative for T4 memory pressure.
    # compilation_config=compilation_config,

    # Stay on a single T4 device; tensor parallelism does not help on a one-GPU Colab runtime.
    tensor_parallel_size=1,

    # For 'RedHatAI/granite-3.1-2b-instruct-quantized.w4a16' default to TRITON_ATTN.
    # This quantized model does not support FLASH_ATTN or FLASHINFER.
    attention_backend="TRITON_ATTN",

    # Try 4-bit inflight quantization on the Colab T4.
    quantization = None, 
    # quantization="bitsandbytes",

    # Disable prefix caching to avoid extra memory overhead on constrained T4 sessions.
    enable_prefix_caching=False,
)


### Test case
In the following, we show a test case with seven candidate passages and a user query.

In [ ]:
case = {
        "query": "Which came first, the invention of the telephone or the light bulb?",
        "documents": [
            [
            "Alexander Graham Bell is credited with inventing the first practical telephone.",
            " He was awarded the U.S. patent for the invention of the telephone on March 7, 1876.",
            " The first successful demonstration of the telephone took place shortly thereafter, when Bell famously called his assistant, saying, 'Mr. Watson, come here, I want to see you.'",
            " Bell’s invention revolutionized communication by allowing people to talk to each other over long distances."
            ],
            [
            "Thomas Edison is widely known for inventing the first commercially practical incandescent light bulb.",
            " Although he did not invent the concept of the light bulb itself, Edison developed a version that was safe, affordable, and long-lasting.",
            " His patent for the electric light bulb was filed in 1879, three years after Bell’s telephone patent.",
            " Edison's innovation led to widespread use of electric lighting and helped usher in the modern electrical age."
            ],
            [
            "Before Edison, several inventors worked on early versions of the light bulb.",
            " Sir Humphry Davy created the first electric arc lamp in the early 1800s, and later inventors like Joseph Swan in Britain improved upon the design.",
            " However, these early bulbs were inefficient or burned out quickly, and it was Edison who perfected the design for everyday use."
            ],
            [
            "The telephone was invented before the practical light bulb.",
            " Bell’s patent for the telephone was issued in 1876, while Edison’s patent for the light bulb was filed in 1879.",
            " Thus, the telephone came first."
            ],
            [
            "Both the telephone and the light bulb are considered groundbreaking inventions of the late 19th century.",
            " The telephone transformed communication, while the light bulb transformed how people lived and worked at night.",
            " Together, they symbolize the rapid technological progress of that era."
            ],
            [
            "Edison and Bell were contemporaries and pioneers of the Second Industrial Revolution.",
            " Their inventions marked major milestones in human history, driving the growth of telecommunications and electrical infrastructure."
            ],
            [
            "In summary, the telephone was invented in 1876 and the light bulb in 1879.",
            " Therefore, the invention of the telephone came first."
            ]
        ]
    }

Next, we apply chat template and obtain the input range using the helper function defined above.<br />
Specifically, as core reranker relies on the aggregated attentions from the user query to each passage, it needs a reference attention baseline for each passage. The authors swap the user query with `'N/A'` and treat the resulting aggregated attention as the normalizing factor for each passage.

In [ ]:
query = case["query"]
documents = case["documents"]
        
# Apply chat template and get ranges
query_text, query_spec = apply_chat_template_and_get_ranges(llm.tokenizer, model, query, documents)
na_text, na_spec = apply_chat_template_and_get_ranges(llm.tokenizer, model, 'N/A', documents)

Finally, we perform the model inference:

In [ ]:
llm.generate(query_text, SamplingParams(temperature=0.1, max_tokens=1),
             save_to_disk=True, run_id="corer-doc")
llm.generate(na_text, SamplingParams(temperature=0.1, max_tokens=1),
             save_to_disk=True, run_id="corer-na")

During the model inference in the previous step, vLLM-Hook has automatically saved selected queries and keys. Now, we can directly call the analyzer to get the passage relevance score and the final ranking of passages:

In [ ]:

stats = llm.analyze(analyzer_spec={'query_spec': query_spec, 'na_spec': na_spec})

Finally we can print out the results as follows:

In [ ]:
print(f"Sorted document IDs and scores by CoRe-Reranking: {stats['ranking']}: {stats['scores']}")